# DeepTruth - Audio Deepfake Detector Training

**Run on Google Colab Pro (A100/T4 GPU)**

## Training Pipeline
1. **Stage 1 - Supervised Warm-up** (frozen Wav2Vec2): Train fusion + heads first.
2. **Stage 2 - Full Fine-tuning** (all layers): End-to-end with layer-wise LR.

**Why no SupCon?** SupCon pretraining caused model collapse — model learned to predict
everything as one class. Removed in favour of direct supervised training.

## Collapse safeguards
- `real_acc` and `fake_acc` tracked every epoch separately
- Training stops automatically if either drops below 10%
- `WeightedRandomSampler` keeps 50/50 class balance
- Extra audio folder support: `extra_real_audio/` and `extra_fake_audio/` on Drive


In [ ]:
# â”€â”€ Cell 1: Mount Drive & Setup â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/DeepTruth'
DATA_DIR     = f'{PROJECT_ROOT}'   # datasets are directly in DeepTruth/
CHECKPOINT_DIR = f'{PROJECT_ROOT}/checkpoints/audio'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)

import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], check=False)

In [ ]:
# â”€â”€ Cell 2: Install Dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys
def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('transformers==4.40.0')
pip('soundfile', 'librosa')
pip('scikit-learn', 'tqdm')

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
#  Cell 3: Extract Archives 
import subprocess, glob, os
from pathlib import Path

DATA = DATA_DIR

# WaveFake
if not Path(f'{DATA}/WaveFake/generated_audio').exists():
    print('Extracting WaveFake...')
    subprocess.run(['unzip', '-q', f'{DATA}/WaveFake/generated_audio.zip', '-d', f'{DATA}/WaveFake/'])
    print('WaveFake done')
else:
    print('[SKIP] WaveFake already extracted')

# ASVspoof2021
asv21_check = Path(f'{DATA}/ASVspoof2021/ASVspoof2021_LA_eval')
if not asv21_check.exists():
    print('Extracting ASVspoof2021...')
    # Try both .tar.gz and .zip
    tar_gz = Path(f'{DATA}/ASVspoof2021/ASVspoof2021_LA_eval.tar.gz')
    zip_f  = Path(f'{DATA}/ASVspoof2021/ASVspoof2021_LA_eval.zip')
    if tar_gz.exists():
        r = subprocess.run(['tar', '-xzf', str(tar_gz), '-C', f'{DATA}/ASVspoof2021/'], capture_output=True)
        if r.returncode != 0:
            print('  tar.gz failed, trying plain tar...')
            subprocess.run(['tar', '-xf', str(tar_gz), '-C', f'{DATA}/ASVspoof2021/'])
    elif zip_f.exists():
        subprocess.run(['unzip', '-q', str(zip_f), '-d', f'{DATA}/ASVspoof2021/'])
    else:
        # Find any archive in the folder
        archives = list(Path(f'{DATA}/ASVspoof2021').glob('*'))
        print(f'  Available files: {[a.name for a in archives]}')
    print('ASVspoof2021 extraction done')
else:
    print('[SKIP] ASVspoof2021 already extracted')

# ASVspoof2019
asv19_check = Path(f'{DATA}/ASVspoof2019/LA')
if not asv19_check.exists():
    print('Extracting ASVspoof2019...')
    # Find archive
    archives = list(Path(f'{DATA}/ASVspoof2019').glob('*.zip')) + list(Path(f'{DATA}/ASVspoof2019').glob('*.tar*'))
    print(f'  Archives found: {[a.name for a in archives]}')
    for arc in archives:
        if arc.suffix == '.zip':
            subprocess.run(['unzip', '-q', str(arc), '-d', f'{DATA}/ASVspoof2019/'])
        else:
            subprocess.run(['tar', '-xf', str(arc), '-C', f'{DATA}/ASVspoof2019/'])
    print('ASVspoof2019 extraction done')
else:
    print('[SKIP] ASVspoof2019 already extracted')

# DFADD
dfadd_zips = sorted(glob.glob(f'{DATA}/DFADD/*.zip'))
if dfadd_zips and not Path(f'{DATA}/DFADD/DATASET_GradTTS').exists():
    print('Extracting DFADD...')
    for z in dfadd_zips:
        print(f'  {os.path.basename(z)}')
        subprocess.run(['unzip', '-q', z, '-d', f'{DATA}/DFADD/'])
    print('DFADD done')
else:
    print('[SKIP] DFADD already extracted')

# MLAAD_2000 — arrow format, no extraction needed
print('[SKIP] MLAAD_2000 is arrow format, no extraction needed')

print('Counting audio files...')
AUDIO_EXTS = {'.wav', '.flac', '.mp3'}
for d in ['WaveFake', 'ASVspoof2021', 'ASVspoof2019', 'DFADD']:
    folder = Path(f'{DATA}/{d}')
    count = sum(1 for f in folder.rglob('*') if f.suffix.lower() in AUDIO_EXTS)
    print(f'  {d}: {count:,} files')


In [ ]:
#  DIAGNOSTIC: Show actual extracted structure 
from pathlib import Path
AUDIO_EXTS = {'.wav', '.flac', '.mp3'}

for d in ['ASVspoof2019', 'ASVspoof2021']:
    folder = Path(DATA_DIR) / d
    count = sum(1 for f in folder.rglob('*') if f.suffix.lower() in AUDIO_EXTS)
    print('=== ' + d + ' (exists=' + str(folder.exists()) + ', audio=' + str(count) + ') ===')
    if not folder.exists():
        continue
    items = sorted(folder.rglob('*'))[:40]
    for item in items:
        rel = item.relative_to(folder)
        kind = 'DIR' if item.is_dir() else item.suffix
        print('  ' + str(rel) + '  (' + kind + ')')


In [ ]:
# â”€â”€ Cell 4: Load Model Architecture â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
from audio_model_arch import (
    DeepTruthAudioV1, DeepTruthAudioLoss,
    SAMPLE_RATE, CLIP_SAMPLES, AUDIO_MANIPULATION_TYPES, N_AUDIO_TYPES
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

with torch.no_grad():
    _m = DeepTruthAudioV1().to(DEVICE)
    _x = torch.randn(2, CLIP_SAMPLES).to(DEVICE)
    _o = _m(_x)
    total = sum(p.numel() for p in _m.parameters()) / 1e6
    print(f'Model OK â€” fake_logit: {_o["fake_logit"].shape}, params: {total:.1f}M')
del _m, _x, _o

In [ ]:
#  Cell 5: Dataset Scanner 
import os, random
import numpy as np
from pathlib import Path
from collections import Counter

TYPE_MAP = {name: i for i, name in enumerate(AUDIO_MANIPULATION_TYPES)}
AUDIO_EXTS = {'.wav', '.flac', '.mp3'}

def get_files(directory, exts=AUDIO_EXTS):
    d = Path(directory)
    if not d.exists(): return []
    return [f for f in d.rglob('*') if f.suffix.lower() in exts]

all_samples = []  # list of (path, binary_label, type_label)

# WaveFake
wf_root = Path(DATA_DIR) / 'WaveFake' / 'generated_audio'
if wf_root.exists():
    for sub in wf_root.iterdir():
        if not sub.is_dir(): continue
        name = sub.name.lower()
        if name == 'real' or 'ljspeech' in name:
            t, b = TYPE_MAP['real'], 0
        elif 'waveglow' in name: t, b = TYPE_MAP['tts_waveglow'], 1
        elif 'hifigan' in name:  t, b = TYPE_MAP['tts_hifigan'], 1
        else:                    t, b = TYPE_MAP['gan_vocoder'], 1
        for f in get_files(sub):
            all_samples.append((str(f), b, t))
    print('WaveFake: ' + str(len(all_samples)) + ' samples')

# ASVspoof2019
asv19_root = Path(DATA_DIR) / 'ASVspoof2019'
asv19 = asv19_root / 'LA'
if not asv19.exists():
    candidates = list(asv19_root.glob('**/LA'))
    if candidates:
        asv19 = candidates[0]
before = len(all_samples)
if asv19.exists():
    for split in ['train', 'dev', 'eval']:
        audio_dir = asv19 / ('ASVspoof2019_LA_' + split) / 'flac'
        proto = asv19 / 'ASVspoof2019_LA_cm_protocols' / ('ASVspoof2019.LA.cm.' + split + '.trl.txt')
        if proto.exists() and audio_dir.exists():
            with open(proto) as pf:
                for line in pf:
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    fpath = audio_dir / (parts[1] + '.flac')
                    if not fpath.exists(): continue
                    binary = 0 if parts[4] == 'bonafide' else 1
                    type_i = TYPE_MAP['real'] if binary == 0 else TYPE_MAP['tts_tacotron']
                    all_samples.append((str(fpath), binary, type_i))
    if len(all_samples) == before:
        for f in get_files(asv19_root):
            all_samples.append((str(f), 1, TYPE_MAP['tts_tacotron']))
print('ASVspoof2019: added ' + str(len(all_samples) - before) + ' samples')

# ASVspoof2021
asv21_root = Path(DATA_DIR) / 'ASVspoof2021'
asv21 = asv21_root / 'ASVspoof2021_LA_eval' / 'flac'
if not asv21.exists():
    candidates = list(asv21_root.glob('**/flac'))
    if candidates:
        asv21 = candidates[0]
before = len(all_samples)
if asv21.exists():
    for f in get_files(asv21):
        all_samples.append((str(f), 1, TYPE_MAP['voice_conversion']))
else:
    for f in get_files(asv21_root):
        all_samples.append((str(f), 1, TYPE_MAP['voice_conversion']))
print('ASVspoof2021: added ' + str(len(all_samples) - before) + ' samples')

# DFADD
dfadd = Path(DATA_DIR) / 'DFADD'
before = len(all_samples)
if dfadd.exists():
    for sub in dfadd.iterdir():
        if not sub.is_dir(): continue
        name = sub.name.lower()
        binary = 0 if 'real' in name or 'original' in name else 1
        type_i = TYPE_MAP['real'] if binary == 0 else TYPE_MAP['diffusion_vocoder']
        for f in get_files(sub):
            all_samples.append((str(f), binary, type_i))
print('DFADD: added ' + str(len(all_samples) - before) + ' samples')

# MLAAD_2000 (arrow format)
mlaad_dir = Path(DATA_DIR) / 'MLAAD_2000'
MLAAD_SAMPLES = []
if mlaad_dir.exists():
    try:
        from datasets import load_from_disk
        ds = load_from_disk(str(mlaad_dir))
        MLAAD_SAMPLES = ds
        print('MLAAD_2000: ' + str(len(ds)) + ' samples loaded (arrow format)')
    except Exception as e:
        print('MLAAD_2000 load error: ' + str(e))

real_count = sum(1 for _, b, _ in all_samples if b == 0)
fake_count = sum(1 for _, b, _ in all_samples if b == 1)
print('File-based samples: ' + str(len(all_samples)))
print('  Real: ' + str(real_count) + '  Fake: ' + str(fake_count))
print('  MLAAD_2000: ' + str(len(MLAAD_SAMPLES)) + ' arrow samples')

type_counts = Counter(AUDIO_MANIPULATION_TYPES[t] for _, _, t in all_samples)
for name, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
    print('  ' + name.ljust(25) + ': ' + str(cnt))

# Optional: extra real audio from Drive (personal recordings)
# Place real recordings in DeepTruth/extra_real_audio/ on Drive.
# Fake audio comes only from the datasets above (WaveFake, ASVspoof, DFADD).
extra_real_dir = Path(DATA_DIR) / 'real_audio'
EXTRA_REAL_REPEAT = 20  # repeat each file x20 so augmentation creates variety
extra_real_files = list(get_files(extra_real_dir))
extra_real_count = len(extra_real_files)
for f in extra_real_files:
    for _ in range(EXTRA_REAL_REPEAT):
        all_samples.append((str(f), 0, TYPE_MAP['real']))
if extra_real_count:
    print(f'Extra real audio: {extra_real_count} files x{EXTRA_REAL_REPEAT} repeats '
          f'= {extra_real_count * EXTRA_REAL_REPEAT} effective samples')
else:
    print('No real_audio folder found (optional).')

real_count = sum(1 for _, b, _ in all_samples if b == 0)
fake_count = sum(1 for _, b, _ in all_samples if b == 1)
print(f'TOTAL: {len(all_samples)} samples  Real: {real_count}  Fake: {fake_count}')


In [ ]:
# â”€â”€ Cell 6: Dataset & DataLoaders â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch, random, numpy as np, soundfile as sf, librosa
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class AudioDataset(Dataset):
    def __init__(self, samples, mlaad_ds=None, train=True):
        self.samples  = samples
        self.mlaad    = mlaad_ds
        self.train    = train
        self.n_file   = len(samples)
        self.n_mlaad  = len(mlaad_ds) if mlaad_ds else 0

    def __len__(self):
        return self.n_file + self.n_mlaad

    def _load_file(self, path):
        try:
            wav, sr = sf.read(path, dtype='float32', always_2d=False)
            if wav.ndim > 1: wav = wav.mean(axis=1)
            if sr != SAMPLE_RATE:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
        except:
            wav = np.zeros(CLIP_SAMPLES, dtype=np.float32)
        if len(wav) >= CLIP_SAMPLES:
            start = random.randint(0, len(wav)-CLIP_SAMPLES) if self.train else 0
            wav = wav[start:start+CLIP_SAMPLES]
        else:
            wav = np.pad(wav, (0, CLIP_SAMPLES-len(wav)))
        return wav

    def _load_mlaad(self, idx):
        row = self.mlaad[idx]
        wav = np.array(row['audio_tensor'], dtype=np.float32).flatten()
        sr  = row['sample_rate']
        if sr != SAMPLE_RATE:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
        if len(wav) >= CLIP_SAMPLES:
            wav = wav[:CLIP_SAMPLES]
        else:
            wav = np.pad(wav, (0, CLIP_SAMPLES-len(wav)))
        label = row['label']
        binary = 0 if label == 'bonafide' else 1
        type_i = TYPE_MAP['real'] if binary == 0 else TYPE_MAP['tts_vits']
        return wav, binary, type_i

    def _augment(self, wav):
        # Gaussian noise
        if random.random() < 0.6:
            noise_level = random.uniform(0.001, 0.015)
            wav = wav + np.random.randn(*wav.shape).astype(np.float32) * noise_level
        # Volume jitter
        if random.random() < 0.6:
            wav = wav * random.uniform(0.5, 1.5)
        # Time shift
        if random.random() < 0.5:
            wav = np.roll(wav, random.randint(-3200, 3200))
        # Random crop (simulate different start point)
        if random.random() < 0.5 and len(wav) > CLIP_SAMPLES:
            start = random.randint(0, len(wav) - CLIP_SAMPLES)
            wav = wav[start:start + CLIP_SAMPLES]
        # Low-level polarity flip (captures phase variations)
        if random.random() < 0.1:
            wav = -wav
        return np.clip(wav, -1.0, 1.0)

    def __getitem__(self, idx):
        if idx < self.n_file:
            path, binary, type_i = self.samples[idx]
            wav = self._load_file(path)
        else:
            wav, binary, type_i = self._load_mlaad(idx - self.n_file)
        if self.train:
            wav = self._augment(wav)
        return (torch.from_numpy(wav),
                torch.tensor(binary, dtype=torch.long),
                torch.tensor(type_i, dtype=torch.long))

# Train/val split
random.seed(42)
shuffled = all_samples[:]
random.shuffle(shuffled)
split = int(len(shuffled) * 0.9)
train_samples = shuffled[:split]
val_samples   = shuffled[split:]

# MLAAD only in train
train_ds = AudioDataset(train_samples, mlaad_ds=MLAAD_SAMPLES if MLAAD_SAMPLES else None, train=True)
val_ds   = AudioDataset(val_samples,   mlaad_ds=None, train=False)

# WeightedRandomSampler
labels = [b for _, b, _ in train_samples]
if MLAAD_SAMPLES:
    labels += [0 if r['label']=='bonafide' else 1 for r in MLAAD_SAMPLES]
class_counts = np.bincount(labels)
weights = 1.0 / class_counts[np.array(labels)]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# DataLoaders â€” num_workers=0 for 12it/s
BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')
wav_b, lab_b, typ_b = next(iter(train_loader))
print(f'Batch: wav={wav_b.shape} labels={lab_b.shape}')
print(f'Label distribution: real={int((lab_b==0).sum())} fake={int((lab_b==1).sum())}')

In [ ]:
# Cell 7 - Master Setup: Instantiate model
import torch
from audio_model_arch import DeepTruthAudioV1, DeepTruthAudioLoss

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

audio_model     = DeepTruthAudioV1(num_fake_types=N_AUDIO_TYPES, dropout=0.3).to(DEVICE)
audio_criterion = DeepTruthAudioLoss(type_weight=0.3, focal_gamma=2.0, label_smoothing=0.1)

total     = sum(p.numel() for p in audio_model.parameters()) / 1e6
trainable = sum(p.numel() for p in audio_model.parameters() if p.requires_grad) / 1e6
print(f'Total: {total:.1f}M  Trainable: {trainable:.1f}M')
if DEVICE.type == 'cuda':
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')


In [ ]:
# Cell 8 - Training utilities
import torch, numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, mode='max'):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.counter   = 0
        self.best      = None
        self.should_stop = False

    def __call__(self, metric):
        if self.best is None:
            self.best = metric
        elif (self.mode == 'max' and metric > self.best + self.min_delta) or \
             (self.mode == 'min' and metric < self.best - self.min_delta):
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


class RealAccGuard:
    """
    Protects real-audio detection accuracy.
    If real_acc drops more than `tolerance` below its peak, rolls back
    the model to the last best checkpoint and stops training.
    """
    def __init__(self, tolerance=0.10):
        self.tolerance  = tolerance   # allow at most 10% drop from peak
        self.peak       = 0.0
        self.should_stop = False

    def __call__(self, real_acc, model, best_ckpt_path):
        import os
        if real_acc > self.peak:
            self.peak = real_acc
        if self.peak > 0 and real_acc < self.peak - self.tolerance:
            print(f'  REAL-ACC GUARD: real_acc={real_acc:.4f} dropped '
                  f'{self.peak - real_acc:.4f} below peak ({self.peak:.4f}).')
            if os.path.exists(best_ckpt_path):
                ckpt = torch.load(best_ckpt_path, map_location='cpu')
                model.load_state_dict(ckpt['model_state'])
                print(f'  Rolled back to best checkpoint (AUC={ckpt["val_auc"]:.4f}).')
            self.should_stop = True
        return self.should_stop


def evaluate_audio(model, loader):
    """Returns auc, acc, real_acc, fake_acc. Both class accuracies tracked to detect collapse."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for wav, labels, type_ids in loader:
            out   = model(wav.to(DEVICE))
            logit = out['fake_logit']
            probs = torch.softmax(logit, dim=1)[:, 1] if logit.shape[1] == 2 \
                    else torch.sigmoid(logit.squeeze(1))
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs > 0.5).astype(int)

    auc  = roc_auc_score(all_labels, all_probs)
    acc  = accuracy_score(all_labels, preds)

    real_mask = all_labels == 0
    fake_mask = all_labels == 1
    real_acc = accuracy_score(all_labels[real_mask], preds[real_mask]) if real_mask.any() else float('nan')
    fake_acc = accuracy_score(all_labels[fake_mask], preds[fake_mask]) if fake_mask.any() else float('nan')

    return {'auc': auc, 'acc': acc, 'real_acc': real_acc, 'fake_acc': fake_acc}


print('Training utilities ready (RealAccGuard enabled).')


In [ ]:
# Cell 9 - Sanity Check
# Verify class balance and initial model output distribution before training.

real_count_tr = sum(1 for _, b, _ in train_samples if b == 0)
fake_count_tr = sum(1 for _, b, _ in train_samples if b == 1)
print(f'Train set: {real_count_tr:,} real  |  {fake_count_tr:,} fake')

print('\nChecking initial model outputs (want ~0.3-0.7 for both classes)...')
audio_model.eval()
sample_real, sample_fake = [], []
checked = 0
with torch.no_grad():
    for wav, labels, _ in val_loader:
        out   = audio_model(wav.to(DEVICE))
        logit = out['fake_logit']
        probs = torch.softmax(logit, dim=1)[:, 1].cpu()
        for p, l in zip(probs, labels):
            (sample_real if l == 0 else sample_fake).append(p.item())
        checked += len(labels)
        if checked >= 256: break

mean_real = sum(sample_real) / max(len(sample_real), 1)
mean_fake = sum(sample_fake) / max(len(sample_fake), 1)
print(f'  Mean fake-prob for REAL audio: {mean_real:.4f}')
print(f'  Mean fake-prob for FAKE audio: {mean_fake:.4f}')

if mean_real > 0.85:
    print('  WARNING: model already predicts real audio as fake — biased init.')
elif mean_fake < 0.2:
    print('  WARNING: model already predicts fake audio as real — biased init.')
else:
    print('  OK: initial outputs look reasonable. Proceed to Stage 1.')


In [ ]:
# Cell 10 - Stage 1: Supervised Warm-up
# Train fusion + heads with Wav2Vec2 FROZEN.
# RealAccGuard rolls back and stops if real detection degrades.
from torch.amp import GradScaler, autocast
from tqdm import tqdm

STAGE1_EPOCHS = 10
STAGE1_LR     = 5e-4
STAGE1_CKPT   = f'{CHECKPOINT_DIR}/stage1_warmup_best.pth'

# Freeze Wav2Vec2 entirely; train only fusion + heads
for p in audio_model.parameters():
    p.requires_grad = False
for name, p in audio_model.named_parameters():
    if any(h in name for h in ['audio_head', 'audio_binary_out', 'audio_type_out', 'audio_fusion']):
        p.requires_grad = True

stage1_params = [p for p in audio_model.parameters() if p.requires_grad]
print(f'Stage 1 trainable: {sum(p.numel() for p in stage1_params)/1e6:.2f}M (wav2vec frozen)')

opt1     = torch.optim.AdamW(stage1_params, lr=STAGE1_LR, weight_decay=1e-4)
scaler   = GradScaler('cuda')
sched1   = torch.optim.lr_scheduler.OneCycleLR(
    opt1, max_lr=STAGE1_LR,
    steps_per_epoch=len(train_loader),
    epochs=STAGE1_EPOCHS, pct_start=0.1
)
early1   = EarlyStopping(patience=4, min_delta=0.005, mode='max')
real_guard1 = RealAccGuard(tolerance=0.10)  # stop if real_acc drops >10% from peak
best_auc = 0.0

for epoch in range(1, STAGE1_EPOCHS + 1):
    audio_model.train()
    total_loss, n_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f'S1 E{epoch}/{STAGE1_EPOCHS}')
    for wav, labels, type_ids in pbar:
        wav, labels, type_ids = wav.to(DEVICE), labels.to(DEVICE), type_ids.to(DEVICE)
        opt1.zero_grad(set_to_none=True)
        with autocast('cuda', dtype=torch.bfloat16):
            out  = audio_model(wav)
            loss, _ = audio_criterion(out, labels, type_ids)
        if torch.isnan(loss): continue
        scaler.scale(loss).backward()
        scaler.unscale_(opt1)
        torch.nn.utils.clip_grad_norm_(stage1_params, 1.0)
        scaler.step(opt1)
        scaler.update()
        sched1.step()
        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    metrics  = evaluate_audio(audio_model, val_loader)
    avg_loss = total_loss / max(n_batches, 1)
    print(f'  S1 E{epoch}: loss={avg_loss:.4f} auc={metrics["auc"]:.4f} '
          f'acc={metrics["acc"]:.4f} '
          f'real_acc={metrics["real_acc"]:.4f} fake_acc={metrics["fake_acc"]:.4f}')

    # Hard collapse guard
    if metrics['fake_acc'] < 0.1:
        print('  COLLAPSE: fake_acc < 10%. Stopping.')
        break

    # Save best before real_acc guard can roll back
    if metrics['auc'] > best_auc:
        best_auc = metrics['auc']
        torch.save({'model_state': audio_model.state_dict(),
                    'epoch': epoch, 'val_auc': best_auc,
                    'real_acc': metrics['real_acc'],
                    'fake_acc': metrics['fake_acc']}, STAGE1_CKPT)
        print(f'    -> Best AUC: {best_auc:.4f} '
              f'(real={metrics["real_acc"]:.4f} fake={metrics["fake_acc"]:.4f}) saved')

    # Real-acc guard: rolls back and stops if real detection degrades
    if real_guard1(metrics['real_acc'], audio_model, STAGE1_CKPT):
        print('  Stopping to protect real audio detection.')
        break

    if early1(metrics['auc']):
        print('Early stopping.')
        break

print(f'Stage 1 complete. Best AUC: {best_auc:.4f}')


In [ ]:
# Cell 11 - Stage 2: Full Fine-tuning
# Very low LR on Wav2Vec2 to preserve learned audio representations.
# RealAccGuard rolls back immediately if real detection degrades.
from torch.amp import GradScaler, autocast
from tqdm import tqdm

STAGE2_EPOCHS = 10
STAGE2_LR     = 3e-5   # conservative — lower than before to preserve real detection
STAGE2_CKPT   = f'{CHECKPOINT_DIR}/stage2_finetune_best.pth'

ckpt1 = torch.load(STAGE1_CKPT, map_location=DEVICE)
audio_model.load_state_dict(ckpt1['model_state'], strict=False)
print(f'Loaded Stage 1 best (AUC={ckpt1["val_auc"]:.4f} '
      f'real={ckpt1["real_acc"]:.4f} fake={ckpt1["fake_acc"]:.4f})')

# Unfreeze all except wav2vec CNN feature extractor
for p in audio_model.parameters(): p.requires_grad = True
for p in audio_model.stream2_wav2vec.wav2vec.feature_extractor.parameters():
    p.requires_grad = False

wav2vec_params = [p for n, p in audio_model.named_parameters()
                  if 'stream2_wav2vec.wav2vec.encoder' in n and p.requires_grad]
other_params   = [p for n, p in audio_model.named_parameters()
                  if 'stream2_wav2vec.wav2vec.encoder' not in n and p.requires_grad]

# Wav2Vec2 encoder gets 10x lower LR — preserves audio understanding
opt2 = torch.optim.AdamW([
    {'params': wav2vec_params, 'lr': STAGE2_LR / 10},  # 3e-6
    {'params': other_params,   'lr': STAGE2_LR},        # 3e-5
], weight_decay=1e-5)
scaler     = GradScaler('cuda')
sched2     = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=STAGE2_EPOCHS, eta_min=1e-7)
early2     = EarlyStopping(patience=5, min_delta=0.002, mode='max')
real_guard2 = RealAccGuard(tolerance=0.08)  # tighter in stage 2 — 8% drop triggers rollback
best_auc   = ckpt1['val_auc']

for epoch in range(1, STAGE2_EPOCHS + 1):
    audio_model.train()
    total_loss, n_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f'S2 E{epoch}/{STAGE2_EPOCHS}')
    for wav, labels, type_ids in pbar:
        wav, labels, type_ids = wav.to(DEVICE), labels.to(DEVICE), type_ids.to(DEVICE)
        opt2.zero_grad(set_to_none=True)
        with autocast('cuda', dtype=torch.bfloat16):
            out  = audio_model(wav)
            loss, _ = audio_criterion(out, labels, type_ids)
        if torch.isnan(loss): continue
        scaler.scale(loss).backward()
        scaler.unscale_(opt2)
        torch.nn.utils.clip_grad_norm_(audio_model.parameters(), 1.0)
        scaler.step(opt2)
        scaler.update()
        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    sched2.step()
    metrics  = evaluate_audio(audio_model, val_loader)
    avg_loss = total_loss / max(n_batches, 1)
    lrs = [f'{g["lr"]:.2e}' for g in opt2.param_groups]
    print(f'  S2 E{epoch}: loss={avg_loss:.4f} auc={metrics["auc"]:.4f} '
          f'acc={metrics["acc"]:.4f} '
          f'real_acc={metrics["real_acc"]:.4f} fake_acc={metrics["fake_acc"]:.4f} '
          f'lrs={lrs}')

    # Hard collapse guard
    if metrics['fake_acc'] < 0.1:
        print('  COLLAPSE: fake_acc < 10%. Stopping.')
        break

    # Save best first so rollback has a good target
    if metrics['auc'] > best_auc:
        best_auc = metrics['auc']
        torch.save({'model_state': audio_model.state_dict(),
                    'epoch': epoch, 'val_auc': best_auc,
                    'real_acc': metrics['real_acc'],
                    'fake_acc': metrics['fake_acc']}, STAGE2_CKPT)
        print(f'    -> Best AUC: {best_auc:.4f} '
              f'(real={metrics["real_acc"]:.4f} fake={metrics["fake_acc"]:.4f}) saved')

    # Real-acc guard: if real detection drops, roll back and stop
    if real_guard2(metrics['real_acc'], audio_model, STAGE2_CKPT):
        print('  Stopping to protect real audio detection.')
        break

    if early2(metrics['auc']):
        print('Early stopping.')
        break

print(f'Stage 2 complete. Best AUC: {best_auc:.4f}')


In [ ]:
# â”€â”€ Cell 11: Stage 3 â€” Full Fine-tune â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch
from torch.amp import GradScaler
from torch.amp import autocast
from tqdm import tqdm

STAGE3_EPOCHS = 10
STAGE3_LR     = 5e-5
STAGE2_CKPT   = f'{CHECKPOINT_DIR}/stage3_finetune.pth'

ckpt2 = torch.load(STAGE2_CKPT, map_location=DEVICE)
audio_model.load_state_dict(ckpt2['model_state'], strict=False)
print(f'Loaded stage2 (AUC={ckpt2["val_auc"]:.4f})')

# Unfreeze all except wav2vec feature extractor
for p in audio_model.parameters(): p.requires_grad = True
for p in audio_model.stream2_wav2vec.wav2vec.feature_extractor.parameters():
    p.requires_grad = False

wav2vec_params = [p for n,p in audio_model.named_parameters()
                  if 'stream2_wav2vec.wav2vec.encoder' in n and p.requires_grad]
other_params   = [p for n,p in audio_model.named_parameters()
                  if 'stream2_wav2vec.wav2vec.encoder' not in n and p.requires_grad]

opt3 = torch.optim.AdamW([
    {'params': wav2vec_params, 'lr': STAGE3_LR/5},
    {'params': other_params,   'lr': STAGE3_LR},
], weight_decay=1e-5)
scaler = GradScaler('cuda')
sched3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=STAGE3_EPOCHS, eta_min=1e-7)

best_auc = ckpt2['val_auc']

for epoch in range(1, STAGE3_EPOCHS + 1):
    audio_model.train()
    total_loss, n_batches, nan_batches = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f'S3 E{epoch}/{STAGE3_EPOCHS}')
    for wav, labels, type_ids in pbar:
        wav      = wav.to(DEVICE, non_blocking=True)
        labels   = labels.to(DEVICE, non_blocking=True)
        type_ids = type_ids.to(DEVICE, non_blocking=True)

        opt3.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            out = audio_model(wav)
            loss, loss_dict = audio_criterion(out, labels, type_ids)

        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(opt3)
        torch.nn.utils.clip_grad_norm_([p for pg in opt3.param_groups for p in pg['params']], 1.0)
        scaler.step(opt3)
        scaler.update()

        total_loss += loss.item()
        n_batches  += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}',
                         main=f'{loss_dict["main"]:.3f}', nan=nan_batches)

    sched3.step()
    avg_loss = total_loss / max(n_batches, 1)
    val_acc, val_auc = evaluate_audio(audio_model, val_loader)
    print(f'  Epoch {epoch}: loss={avg_loss:.4f}  val_acc={val_acc:.4f}  val_AUC={val_auc:.4f}  nan={nan_batches}')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save({'model_state': audio_model.state_dict(), 'epoch': epoch,
                    'val_auc': val_auc, 'val_acc': val_acc, 'num_fake_types': N_AUDIO_TYPES,
                    'fake_type_map': {n:i for i,n in enumerate(AUDIO_MANIPULATION_TYPES)},
                    'sample_rate': SAMPLE_RATE, 'clip_samples': CLIP_SAMPLES},
                   STAGE2_CKPT)
        print(f'  [SAVED] stage3 best (AUC={best_auc:.4f})')

print(f'Stage 3 done. Best AUC: {best_auc:.4f}')

In [ ]:
# â”€â”€ Cell 12: Test Evaluation & Save Final Model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch, numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

ckpt3 = torch.load(STAGE2_CKPT, map_location=DEVICE)
audio_model.load_state_dict(ckpt3['model_state'], strict=False)
audio_model.eval()
print(f'Loaded best model (epoch={ckpt3["epoch"]}, AUC={ckpt3["val_auc"]:.4f})')

all_probs, all_labels = [], []
with torch.no_grad():
    for wav, labels, _ in val_loader:
        out   = audio_model(wav.to(DEVICE))
        logit = out['fake_logit']
        probs = torch.softmax(logit, dim=1)[:, 1] if logit.shape[1]==2 else torch.sigmoid(logit.squeeze(1))
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
test_auc = roc_auc_score(all_labels, all_probs)
test_acc = accuracy_score(all_labels, (all_probs > 0.5).astype(int))
print(f'Test AUROC: {test_auc:.4f}')
print(f'Test Acc:   {test_acc:.4f}')
print(classification_report(all_labels, (all_probs>0.5).astype(int), target_names=['Real','Fake']))

FINAL = f'{PROJECT_ROOT}/models/deeptruth_audio_model_final.pth'
os.makedirs(f'{PROJECT_ROOT}/models', exist_ok=True)
torch.save({'model_state': audio_model.state_dict(), 'num_fake_types': N_AUDIO_TYPES,
            'fake_type_map': {n:i for i,n in enumerate(AUDIO_MANIPULATION_TYPES)},
            'sample_rate': SAMPLE_RATE, 'clip_samples': CLIP_SAMPLES,
            'test_auroc': test_auc, 'test_acc': test_acc}, FINAL)
print(f'Saved -> {FINAL}')

from google.colab import files
files.download(FINAL)